### Elastic Net Regression

Elastic Net is a regularized linear regression technique that combines **L1 (Lasso)** and **L2 (Ridge)** penalties. It controls model complexity by penalizing large coefficients and can also set some coefficients exactly to zero.

Elastic Net is useful when a dataset contains many features, especially when some features are correlated. It combines Lasso's ability to perform feature selection with Ridge's tendency to distribute coefficient weights across correlated features.

Two important parameters are:
- `alpha`: controls the overall strength of regularization.
- `l1_ratio`: controls the balance between L1 and L2 penalties. A value of `1` corresponds to Lasso, while `0` corresponds to Ridge.

In [1]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression,Ridge,Lasso,ElasticNet
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

In [2]:
X,y = load_diabetes(return_X_y=True)

In [3]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=2)

#### Linear Regression (Baseline)

Linear Regression is trained without a regularization penalty. Its R² score provides a baseline for comparison with the regularized models.

R² measures how much of the variation in the target is explained by the model. A higher R² generally indicates a better fit on the evaluated data, but it should be interpreted alongside other evaluation metrics and validation results.

In [4]:
# Linear Regression
reg = LinearRegression()
reg.fit(X_train,y_train)
y_pred = reg.predict(X_test)
r2_score(y_test,y_pred)

0.4399338661568968

#### Ridge Regression (L2)

Ridge Regression adds an L2 penalty, which is based on the sum of squared coefficients. It shrinks coefficients toward zero but generally does not make them exactly zero.

In this notebook, `alpha=0.1` controls the regularization strength. Ridge can be useful when several features contribute to the target and some features are correlated.

In [5]:
# Ridge 
reg = Ridge(alpha=0.1)
reg.fit(X_train,y_train)
y_pred = reg.predict(X_test)
r2_score(y_test,y_pred)

0.45199494197195456

#### Lasso Regression (L1)

Lasso Regression adds an L1 penalty, based on the sum of the absolute values of the coefficients. It can shrink some coefficients exactly to zero, effectively removing those features from the model.

In this notebook, `alpha=0.01` controls the regularization strength. Lasso is useful when a simpler model with automatic feature selection is desired.

In [6]:
# Lasso
reg = Lasso(alpha=0.01)
reg.fit(X_train,y_train)
y_pred = reg.predict(X_test)
r2_score(y_test,y_pred)

0.44111855963110613

#### Elastic Net (L1 + L2)

Elastic Net combines the L1 and L2 penalties. It can set some coefficients to zero like Lasso, while also using the L2 penalty to help manage correlated features.

In this notebook:
- `alpha=0.005` controls the overall regularization strength.
- `l1_ratio=0.9` gives more weight to the L1 penalty than the L2 penalty.

The model is evaluated using the R² score on the test set.

In [7]:
# ElasticNet
reg = ElasticNet(alpha=0.005,l1_ratio=0.9)
reg.fit(X_train,y_train)
y_pred = reg.predict(X_test)
r2_score(y_test,y_pred)

0.4531474541554823

#### Compare All Models

The following cell trains each model with the same train/test split and displays their test-set R² scores together. This makes it easier to compare the baseline and regularized models.

In [8]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import r2_score
import pandas as pd

models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=0.1),
    "Lasso": Lasso(alpha=0.01, max_iter=10000),
    "Elastic Net": ElasticNet(alpha=0.005, l1_ratio=0.9, max_iter=10000)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    results.append({
        "Model": name,
        "Test R²": r2_score(y_test, predictions)
    })

pd.DataFrame(results).sort_values("Test R²", ascending=False).reset_index(drop=True)

,Model,Test R²
0,Elastic Net,0.453147
1,Ridge,0.451995
2,Lasso,0.441119
3,Linear Regression,0.439934


#### Elastic Net Objective Function

Elastic Net minimizes the prediction error plus a combination of L1 and L2 penalties:

$$
J(\mathbf{w}) =
\frac{1}{2n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2
+
\alpha\left(
l1\_ratio\sum_{j=1}^{p}|w_j|
+
\frac{1-l1\_ratio}{2}\sum_{j=1}^{p}w_j^2
\right)
$$

Where:
- $n$ is the number of training observations.
- $p$ is the number of features.
- $w_j$ is the coefficient of feature $j$.
- `alpha` controls the overall penalty strength.
- `l1_ratio` controls the balance between L1 and L2 regularization.

#### Ridge vs Lasso vs Elastic Net

| Aspect | Ridge | Lasso | Elastic Net |
|:--|:--|:--|:--|
| Penalty | L2 | L1 | L1 + L2 |
| Shrinks coefficients | Yes | Yes | Yes |
| Can set coefficients exactly to zero | Generally no | Yes | Yes |
| Feature selection | No automatic selection | Yes | Yes |
| Correlated features | Often handles them well by sharing weights | May select one and suppress others | Combines Lasso selection with Ridge-style shrinkage |

#### Key Takeaways

- **Ridge** is useful when many features contribute to the prediction, including correlated features.
- **Lasso** can produce sparse models by setting some coefficients to zero.
- **Elastic Net** combines both penalties and can be a good choice when features are correlated and feature selection is also desired.
- `alpha` controls the overall regularization strength, while `l1_ratio` controls the L1/L2 balance.
- The best settings should be selected using cross-validation on the training data. Use the test set only for final evaluation.